# Notebook to open all OceanSpy datasets on SciServer-ceph and display basic information about them.

If the datasets fail to open, make sure that your SciServer container includes the Poseidon (ceph) and Ocean Circulation (ceph) data volumes.

See also `ListDatasets-SciServer-ceph_xarray.ipynb` and `ListDatasets-SciServer-ceph_xmitgcm.ipynb` to open datasets without using Oceanspy.

TWNH Jan '23, Jun '24, Jun '26

In [1]:
import os
from pathlib import Path
import time
import traceback
import yaml
import pandas as pd

import oceanspy as ospy

In [2]:
USE_LOCAL_CATALOGS = True

LOCAL_CATALOG_DIR = "/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs"

if USE_LOCAL_CATALOGS:
    os.environ["OCEANSPY_CATALOG_DIR"] = LOCAL_CATALOG_DIR
else:
    os.environ.pop("OCEANSPY_CATALOG_DIR", None)

print("USE_LOCAL_CATALOGS =", USE_LOCAL_CATALOGS)
print("OCEANSPY_CATALOG_DIR =", os.environ.get("OCEANSPY_CATALOG_DIR"))
print("OceanSpy version =", ospy.__version__)
print("OceanSpy file =", ospy.__file__)

USE_LOCAL_CATALOGS = True
OCEANSPY_CATALOG_DIR = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs
OceanSpy version = 0.1.dev1246+g9a5e7f07c
OceanSpy file = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/oceanspy/__init__.py


In [3]:
if USE_LOCAL_CATALOGS:
    datasets_list_file = Path(LOCAL_CATALOG_DIR) / "datasets_list.yaml"
    with open(datasets_list_file, "r") as f:
        datasets_list = yaml.safe_load(f)
else:
    import urllib.request as _urllib
    url = "https://raw.githubusercontent.com/hainegroup/oceanspy/main/sciserver_catalogs/datasets_list.yaml"
    with _urllib.urlopen(url) as f:
        datasets_list = yaml.safe_load(f)

SCISERVER_DATASETS = datasets_list["datasets"]["sciserver"]

print(f"Found {len(SCISERVER_DATASETS)} datasets")
print(SCISERVER_DATASETS)

Found 17 datasets
['get_started', 'IGPwinter', 'IGPyearlong', 'EGshelfIIseas2km_ASR_full', 'EGshelfIIseas2km_ASR_crop', 'EGshelfIIseas2km_ERAI_6H', 'EGshelfIIseas2km_ERAI_1D', 'EGshelfSJsec500m_3H_hydro', 'EGshelfSJsec500m_6H_hydro', 'EGshelfSJsec500m_3H_NONhydro', 'EGshelfSJsec500m_6H_NONhydro', 'Arctic_Control', 'KangerFjord', 'HYCOM', 'ECCO', 'daily_ecco', 'ETOPO']


In [4]:
def ocean_dataset_summary(od):
    """
    Return a compact summary dictionary for an OceanDataset.
    """
    ds = od._ds if hasattr(od, "_ds") else od.dataset

    summary = {
        "nbytes_TB": ds.nbytes * 1e-12,
        "ndims": len(ds.dims),
        "nvars": len(ds.data_vars),
        "ncoords": len(ds.coords),
        "dims": dict(ds.sizes),
    }

    try:
        summary["chunks"] = {k: tuple(v) for k, v in ds.chunksizes.items()}
    except Exception:
        summary["chunks"] = None

    return summary


def test_open(dataset_name):
    """
    Open one dataset through OceanSpy and return:
      - result dict
      - OceanDataset object or None
    """
    t0 = time.perf_counter()

    result = {
        "dataset": dataset_name,
        "ok": False,
        "open_time_s": None,
        "nbytes_TB": None,
        "ndims": None,
        "nvars": None,
        "ncoords": None,
        "dims": None,
        "chunks": None,
        "error_type": None,
        "error_msg": None,
    }

    try:
        od = ospy.open_oceandataset.from_catalog(dataset_name)
        dt = time.perf_counter() - t0
        summary = ocean_dataset_summary(od)

        result.update({
            "ok": True,
            "open_time_s": dt,
            "nbytes_TB": summary["nbytes_TB"],
            "ndims": summary["ndims"],
            "nvars": summary["nvars"],
            "ncoords": summary["ncoords"],
            "dims": summary["dims"],
            "chunks": summary["chunks"],
        })

        return result, od

    except Exception as e:
        dt = time.perf_counter() - t0
        result.update({
            "ok": False,
            "open_time_s": dt,
            "error_type": type(e).__name__,
            "error_msg": str(e),
        })
        return result, None

In [5]:
results = []
failures = {}

for dsname in SCISERVER_DATASETS:
    print(f"\nOpening: {dsname}")
    res, od = test_open(dsname)
    results.append(res)

    if res["ok"]:
        print(f"  OK in {res['open_time_s']:.2f} s")
        print(f"  size: {res['nbytes_TB']:.3f} TB")
        print(f"  dims: {res['dims']}")
        print(f"  vars: {res['nvars']}")
    else:
        print(f"  FAIL in {res['open_time_s']:.2f} s")
        print(f"  {res['error_type']}: {res['error_msg']}")
        failures[dsname] = res


Opening: get_started
Opening get_started.
DEBUG: xarray_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xarray.yaml
DEBUG: xmitgcm_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xmitgcm.yaml
DEBUG: intake_switch = True
DEBUG: entries = ['grd_get_started', 'fld_get_started', 'avg_get_started']
Small cutout from EGshelfIIseas2km_ASR_crop.
Citation:
 * Almansi et al., 2020 - GRL.
See also:
 * EGshelfIIseas2km_ASR_full: Full domain without variables to close budgets.
 * EGshelfIIseas2km_ASR_crop: Cropped domain with variables to close budgets.
  OK in 0.77 s
  size: 0.001 TB
  dims: {'Zp1': 56, 'Z': 55, 'Y': 154, 'X': 207, 'Xp1': 208, 'Yp1': 155, 'Zu': 55, 'Zl': 55, 'time': 4, 'time_midp': 3}
  vars: 85

Opening: IGPwinter
Opening IGPwinter.
DEBUG: xarray_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oc

/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/oceanspy/open_oceandataset.py:243: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'X' ('X',) The recommendation is to set join explicitly for this case.
  ds = _xr.merge(datasets)
/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/oceanspy/open_oceandataset.py:243: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'Xp1' ('Xp1',) The recommendation is to set join explicitly for this case.
  ds 

High-resolution (~2km) numerical simulation covering the east Greenland shelf (EGshelf),
and the Iceland and Irminger Seas (IIseas) forced by the Arctic System Reanalysis (ASR).
Citation:
 * Almansi et al., 2020 - GRL.
Characteristics:
 * crop: Cropped domain with variables to close budgets.
See also:
 * EGshelfIIseas2km_ASR_full: Full domain without variables to close budgets.
  OK in 96.27 s
  size: 1.630 TB
  dims: {'X': 414, 'Xp1': 416, 'Y': 308, 'Yp1': 310, 'Z': 55, 'Zl': 55, 'Zp1': 217, 'Zu': 216, 'time': 1464, 'time_midp': 1463}
  vars: 89

Opening: EGshelfIIseas2km_ERAI_6H
Opening EGshelfIIseas2km_ERAI_6H.
DEBUG: xarray_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xarray.yaml
DEBUG: xmitgcm_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xmitgcm.yaml
DEBUG: intake_switch = True
DEBUG: entries = ['grd_EGshelfIIseas2km_ERAI_6H', 'f

/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/oceanspy/open_oceandataset.py:243: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  ds = _xr.merge(datasets)
/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/oceanspy/open_oceandataset.py:243: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set comp

High-resolution (~2km) numerical simulation covering the east Greenland shelf (EGshelf),
and the Iceland and Irminger Seas (IIseas) forced by ERA-Interim.
Citation:
 * Almansi et al., 2017 - JPO.
Characteristics:
 * 1D: 1-day resolution with sea ice and external forcing variables.
See also:
 * EGshelfIIseas2km_ERAI_6H: 6-hour resolution without sea ice and external forcing variables.
  OK in 113.93 s
  size: 0.010 TB
  dims: {'time': 92, 'time_midp': 91, 'X': 960, 'Xp1': 961, 'Y': 880, 'Yp1': 881, 'Z': 216, 'Zl': 216, 'Zp1': 217, 'Zu': 216}
  vars: 30

Opening: EGshelfSJsec500m_3H_hydro
Opening EGshelfSJsec500m_3H_hydro.
DEBUG: xarray_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xarray.yaml
DEBUG: xmitgcm_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xmitgcm.yaml
DEBUG: intake_switch = False
DEBUG: entries = ['EGshelfSJsec500m_3H_hydro

/home/idies/mambaforge/envs/Oceanography/lib/python3.12/site-packages/intake/readers/readers.py:1334: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  return open_mfdataset(ofs, **kw)


ECCO v4r4 3D dataset, ocean simulations on LLC90 grid
  OK in 25.45 s
  size: 5.040 TB
  dims: {'Zp1': 51, 'Yp1': 90, 'Xp1': 90, 'Z': 50, 'Y': 90, 'Zu': 50, 'X': 90, 'Zl': 50, 'face': 13, 'time_midp': 9496, 'time': 9497}
  vars: 35

Opening: ETOPO
Opening ETOPO.
DEBUG: xarray_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xarray.yaml
DEBUG: xmitgcm_url = /home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xmitgcm.yaml
DEBUG: intake_switch = True
DEBUG: entries = ['ETOPO']
The ice surface version of ETOPO
Citation:
 * 10.25921/fd45-gt74.
  OK in 0.87 s
  size: 0.002 TB
  dims: {'Y': 10801, 'X': 21601}
  vars: 1


In [6]:
df = pd.DataFrame(results)

df = df[
    [
        "dataset", "ok", "open_time_s", "nbytes_TB",
        "ndims", "nvars", "ncoords", "dims", "chunks",
        "error_type", "error_msg"
    ]
]

df

,dataset,ok,open_time_s,nbytes_TB,ndims,nvars,ncoords,dims,chunks,error_type,error_msg
0,get_started,True,0.769387,0.001007,10,85,18,"{'Zp1': 56, 'Z': 55, 'Y': 154, 'X': 207, 'Xp1'...","{'Zp1': (56,), 'Z': (55,), 'Y': (154,), 'X': (...",None,None
1,IGPwinter,True,0.728682,10.122228,10,102,18,"{'time_midp': 359, 'Zl': 216, 'Y': 880, 'X': 9...","{'time_midp': (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1...",None,None
2,IGPyearlong,True,0.615395,43.241105,10,104,18,"{'time_midp': 1459, 'Zl': 216, 'Y': 880, 'X': ...","{'time_midp': (2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2...",None,None
3,EGshelfIIseas2km_ASR_full,True,1.015090,17.516920,10,75,18,"{'Y': 880, 'X': 960, 'time': 1464, 'Z': 216, '...","{'Y': (220, 220, 220, 220), 'X': (240, 240, 24...",None,None
4,EGshelfIIseas2km_ASR_crop,True,96.269986,1.629665,10,89,18,"{'X': 414, 'Xp1': 416, 'Y': 308, 'Yp1': 310, '...","{'Y': (77, 77, 77, 77), 'X': (103, 103, 103, 1...",None,None
5,EGshelfIIseas2km_ERAI_6H,True,9.117062,15.088116,10,41,18,"{'Y': 880, 'X': 960, 'time': 1464, 'Z': 216, '...","{'Y': (220, 220, 220, 220), 'X': (240, 240, 24...",None,None
6,EGshelfIIseas2km_ERAI_1D,True,113.933828,0.009527,10,30,18,"{'time': 92, 'time_midp': 91, 'X': 960, 'Xp1':...","{'Y': (220, 220, 220, 220), 'X': (240, 240, 24...",None,None
7,EGshelfSJsec500m_3H_hydro,True,1023.687070,1.270007,10,26,18,"{'X': 725, 'Y': 605, 'Xp1': 725, 'Yp1': 605, '...","{'Y': (605,), 'X': (725,), 'Yp1': (605,), 'Xp1...",None,None
8,EGshelfSJsec500m_6H_hydro,True,902.177025,0.640408,10,35,18,"{'X': 725, 'Y': 605, 'Xp1': 725, 'Yp1': 605, '...","{'Y': (605,), 'X': (725,), 'Yp1': (605,), 'Xp1...",None,None
9,EGshelfSJsec500m_3H_NONhydro,True,1204.236455,1.278534,10,26,18,"{'X': 725, 'Y': 605, 'Xp1': 725, 'Yp1': 605, '...","{'Y': (605,), 'X': (725,), 'Yp1': (605,), 'Xp1...",None,None


In [7]:
print("Total datasets tested:", len(df))
print("Succeeded:", int(df["ok"].sum()))
print("Failed:", int((~df["ok"]).sum()))

print("\nTiming summary for successful opens (s):")
display(df.loc[df["ok"], "open_time_s"].describe())

print("\nLargest successful datasets by size:")
display(
    df.loc[df["ok"]]
      .sort_values("nbytes_TB", ascending=False)
      .head(10)[["dataset", "open_time_s", "nbytes_TB", "ndims", "nvars"]]
)

Total datasets tested: 17
Succeeded: 17
Failed: 0

Timing summary for successful opens (s):


count      17.000000
mean      393.262356
std       676.146646
min         0.615395
25%         1.015090
50%         9.117062
75%       878.455521
max      2416.017126
Name: open_time_s, dtype: float64


Largest successful datasets by size:


,dataset,open_time_s,nbytes_TB,ndims,nvars
2,IGPyearlong,0.615395,43.241105,10,104
3,EGshelfIIseas2km_ASR_full,1.015090,17.516920,10,75
5,EGshelfIIseas2km_ERAI_6H,9.117062,15.088116,10,41
1,IGPwinter,0.728682,10.122228,10,102
15,daily_ecco,25.453091,5.040214,11,35
4,EGshelfIIseas2km_ASR_crop,96.269986,1.629665,10,89
13,HYCOM,1.284756,1.281436,5,5
9,EGshelfSJsec500m_3H_NONhydro,1204.236455,1.278534,10,26
7,EGshelfSJsec500m_3H_hydro,1023.687070,1.270007,10,26
10,EGshelfSJsec500m_6H_NONhydro,878.455521,0.640408,10,35


In [8]:
failed_df = df.loc[~df["ok"]].copy()

if len(failed_df) == 0:
    print("No failures.")
else:
    print("Failed datasets:")
    display(
        failed_df[["dataset", "open_time_s", "error_type", "error_msg"]]
    )

No failures.


In [9]:
def debug_open(dataset_name):
    print(f"\n=== Debug open: {dataset_name} ===")
    t0 = time.perf_counter()
    try:
        od = ospy.open_oceandataset.from_catalog(dataset_name)
        print(f"OK in {time.perf_counter() - t0:.2f} s")
        print(ocean_dataset_summary(od))
    except Exception:
        print(f"FAILED after {time.perf_counter() - t0:.2f} s")
        traceback.print_exc()

In [10]:
# debug_open("KangerFjord")